# Download assemblies of Cryptococcus species complexes from NCBI and compute pairwise whole-genome similarity using sourmash for ANI computation

## ____________________________________________________________________________________________

### Setup libraries, configuration, and directories

In [ ]:
## Import libraries

import sourmash
import subprocess
import tempfile
import pandas as pd
import numpy as np
import os
from multiprocessing import Pool
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from functools import partial
import re
import time
import random
import logging
import shutil

In [ ]:
## Setup directories and configuration variables


# Configuration variables
K_VALUES = [16, 21, 31, 51]  # Using uppercase for constants
SCALED = 1000
DATASETS_TOOL = "/path/to/your/datasets_tool/datasets"
SOURMASH_THREADS = 32
DOWNLOAD_THREADS = 4
MAX_RETRIES = 5
MIN_DELAY = 1.2
MAX_DELAY = 3.5
SIGNATURE_TIMEOUT = 1800

# Define working directory
BASE_DIR = Path("/path/to/your/analysis_directory/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced")
os.chdir(BASE_DIR)


# Create main directories
ASSEMBLY_DIR = BASE_DIR / "BUSCO_CryptococcusReduced" / "genomes"  # reuse assemblies already downloaded for the BUSCO pipeline
TREE_DIR = BASE_DIR / "SourmashTrees"
SOURMASH_DIR = BASE_DIR / "SourmashSignatures"


# Create directories (ASSEMBLY_DIR is shared with the BUSCO pipeline and
# expected to already exist; only create this notebook's own output dirs)
for directory in [TREE_DIR, SOURMASH_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
## Define the accessions used in this analysis

# Read in a csv with filenames of accessions to consider for this analysis (modify as needed, best case scenario is just load a list of the accessions)
ACCESSIONS = pd.read_csv('/path/to/your/analysis_directory/ForGeneTrees/RedoGeneTrees_AlignThenConcatenate/IntermediateFiles/Cryptococcus_Reduced_IF/BUSCO_Trees_pipeline/Over80CompleteAssemblies.txt', header=None)
# Extract just the accession portion of the filenames
ACCESSIONS = ACCESSIONS[0].str.extract(r'(GCA_\d+\.\d+)')[0].tolist()

### Define some necessary functions

In [ ]:
## Download a genome using NCBI datasets tool, with some code for retrying failed downloads and delaying downloads to attempt to prevent NCBI throttling

def download_genome(accession, retries=MAX_RETRIES):
    """Download a single genome with retries."""
    for attempt in range(retries):
        try:
            cmd = f"{DATASETS_TOOL} download genome accession {accession} --filename {ASSEMBLY_DIR}/{accession}.zip"
            subprocess.run(cmd, shell=True, check=True, capture_output=True)
            logging.info(f"Downloaded {accession}")
            return True
        except subprocess.CalledProcessError as e:
            logging.warning(f"Failed to download {accession}, attempt {attempt + 1}/{retries}: {e}")
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
    logging.error(f"Failed to download {accession} after {retries} attempts")
    return False

In [ ]:
## For a downloaded NCBI assembly folder, unzip and move the actual assembly

def extract_and_organize(accession):
    """Extract and organize downloaded genome files."""
    zip_path = ASSEMBLY_DIR / f"{accession}.zip"
    try:
        subprocess.run(f"unzip -o {zip_path} -d {ASSEMBLY_DIR}/{accession}", shell=True, check=True)
        for fna in (ASSEMBLY_DIR / accession).glob("**/*_genomic.fna"):
            shutil.move(str(fna), ASSEMBLY_DIR / f"{accession}_{fna.stem}.fna")
        shutil.rmtree(ASSEMBLY_DIR / accession)
        zip_path.unlink()
    except subprocess.CalledProcessError as e:
        logging.error(f"Failed to extract/organize {accession}: {e}")

In [ ]:
# Compute ANI by Jaccard and Max-Containment
def compute_ani(k, ordered_accessions):
    """Compute Jaccard and max-containment ANI for all signatures in a k-mer directory."""
    k_dir = SOURMASH_DIR / f"k{k}"
    sig_list = k_dir / "sig_list.txt"
    
    # Write signature files in the order of ordered_accessions
    with open(sig_list, "w") as f:
        for acc in ordered_accessions:
            sig_files = list(k_dir.glob(f"{acc}*_k{k}.sig"))
            if sig_files:
                f.write(f"{sig_files[0]}\n")
            else:
                logging.warning(f"No signature file found for accession {acc} with k={k}")
    
    # Compute Jaccard ANI
    jaccard_cmd = f"sourmash compare --from-file {sig_list} -k {k} --ani -o {k_dir}/ani_results_jaccard.npy"
    subprocess.run(jaccard_cmd, shell=True, check=True)
    logging.info(f"Computed Jaccard ANI for k={k}")
    
    # Compute max-containment ANI
    containment_cmd = f"sourmash compare --from-file {sig_list} -k {k} --max-containment --ani -o {k_dir}/ani_results_maxcontainment.npy"
    subprocess.run(containment_cmd, shell=True, check=True)
    logging.info(f"Computed max-containment ANI for k={k}")

In [ ]:
# Compute signature for genome assembly
def compute_signature(fna_file, k):
    """Compute sourmash signature for a given k-mer size with robust error handling."""
    sig_path = SOURMASH_DIR / f"k{k}" / f"{fna_file.stem}_k{k}.sig"
    
    # Check if file exists and has content
    if not fna_file.exists():
        logging.error(f"FASTA file does not exist: {fna_file}")
        return
    if fna_file.stat().st_size == 0:
        logging.error(f"FASTA file is empty: {fna_file}")
        return
    
    # Check if signature already exists
    if sig_path.exists():
        logging.info(f"Signature already exists for {fna_file} with k={k}")
        return
    
    cmd = f"sourmash sketch dna -p k={k},scaled={SCALED} {fna_file} -o {sig_path}"
    
    try:
        # Run with more detailed output
        result = subprocess.run(
            cmd, 
            shell=True, 
            check=True, 
            capture_output=True, 
            text=True,
            timeout=SIGNATURE_TIMEOUT
        )
        
        # Verify the signature was created
        if sig_path.exists() and sig_path.stat().st_size > 0:
            logging.info(f"✓ Successfully created signature for {fna_file} with k={k}")
        else:
            logging.warning(f"Signature file created but appears empty for {fna_file}")
            
    except subprocess.CalledProcessError as e:
        logging.error(f"Failed to compute signature for {fna_file} with k={k}")
        logging.error(f"Command: {cmd}")
        logging.error(f"Exit code: {e.returncode}")
        
        # Try to parse common sourmash errors
        error_msg = e.stderr.lower()
        if "memory" in error_msg:
            logging.error("Possible memory issue - try reducing SCALED parameter")
        elif "kmer size" in error_msg or "k=" in error_msg:
            logging.error(f"Possible issue with k-mer size k={k}")
        elif "empty" in error_msg or "no sequences" in error_msg:
            logging.error("FASTA file might be empty or contain no valid sequences")
        else:
            logging.error(f"Full stderr: {e.stderr[:500]}...")  # First 500 chars
        
        # Clean up failed signature file if it exists
        if sig_path.exists():
            sig_path.unlink()
            
    except subprocess.TimeoutExpired as e:
        logging.error(f"Timeout computing signature for {fna_file} with k={k}")
        # Clean up partial files
        if sig_path.exists():
            sig_path.unlink()

def compute_comprehensive_ani(k, accessions):
    """Compute and save all pairwise ANI comparisons for a given k-value."""
    k_dir = SOURMASH_DIR / f"k{k}"
    
    # Create signature list file
    sig_list = k_dir / "all_signatures.txt"
    with open(sig_list, "w") as f:
        for acc in accessions:
            sig_files = list(k_dir.glob(f"{acc}*_k{k}.sig"))
            if sig_files:
                f.write(f"{sig_files[0]}\n")
            else:
                logging.warning(f"Missing signature for {acc} at k={k}")
    
    # Compute Jaccard ANI matrix
    jaccard_out = k_dir / "comprehensive_jaccard.npy"
    jaccard_cmd = f"sourmash compare --from-file {sig_list} -k {k} --ani -o {jaccard_out}"
    subprocess.run(jaccard_cmd, shell=True, check=True)
    
    # Compute Max-Containment ANI matrix
    containment_out = k_dir / "comprehensive_maxcontainment.npy"
    containment_cmd = f"sourmash compare --from-file {sig_list} -k {k} --max-containment --ani -o {containment_out}"
    subprocess.run(containment_cmd, shell=True, check=True)
    
    # Save accession list for reference
    with open(k_dir / "accession_order.txt", "w") as f:
        f.write("\n".join(accessions))
    
    logging.info(f"Completed comprehensive ANI calculations for k={k}")

In [ ]:
# Main processing function for computing comprehensive pairwise ANI for all assemblies using all k-values and both ANI methods

def Download_And_ComprehensiveANI(accessions):
    """Compute all pairwise ANI comparisons for all accessions and k-values."""

    
    # Only download/extract accessions not already present in ASSEMBLY_DIR
    # (this analysis reuses the assemblies already downloaded for the BUSCO
    # pipeline against the same accession set)
    missing_accessions = [acc for acc in accessions if not list(ASSEMBLY_DIR.glob(f"{acc}*.fna"))]

    if missing_accessions:
        logging.info(f"Downloading {len(missing_accessions)} missing assemblies using {DOWNLOAD_THREADS} threads...")
        with ThreadPoolExecutor(max_workers=DOWNLOAD_THREADS) as executor:
            executor.map(download_genome, missing_accessions)

        logging.info("Extracting assemblies...")
        for accession in missing_accessions:
            extract_and_organize(accession)
    else:
        logging.info("All assemblies already present, skipping download.")
    
    # Get all successfully extracted assemblies
    fna_files = list(ASSEMBLY_DIR.glob("*.fna"))
    logging.info(f"Found {len(fna_files)}/{len(accessions)} assemblies ready for processing")
    
    # Create k-value directories if needed
    for k in K_VALUES:
        (SOURMASH_DIR / f"k{k}").mkdir(exist_ok=True)
    
    # Compute signatures for all k-values
    logging.info("Computing signatures...")
    with ThreadPoolExecutor(max_workers=SOURMASH_THREADS) as executor:
        for fna_file in fna_files:
            for k in K_VALUES:
                executor.submit(compute_signature, fna_file, k)

    

    
    # Compute comprehensive ANI matrices for each k
    logging.info("Computing comprehensive ANI matrices...")
    for k in K_VALUES:
        compute_comprehensive_ani(k, accessions)



In [ ]:
## Compute comprehensive ANI comparisons among all assemblies at different k's and ANI estimations

Download_And_ComprehensiveANI(ACCESSIONS)